# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zainabaon/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [10]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/zainabaon/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    os.chdir("/content")
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

os.makedirs("work/outputs", exist_ok=True)
import pandas as pd
import numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(df.shape)

(30000, 45)


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
df["is_stale"] = df["days_since_last_update"] >= 180
df["is_visible"] = df["impressions_90d"] >= 500
df["baseline_score"] = df["is_stale"].astype(int) * df["is_visible"].astype(int) * df["impressions_90d"]

df["reason_code"] = "stale_visible_page"
df["action"] = "review_for_refresh"
df.loc[~(df["is_stale"] & df["is_visible"]), ["reason_code", "action"]] = ["no_action_needed", "monitor"]

queue = df.sort_values("baseline_score", ascending=False)[
    ["content_id", "client_id", "baseline_score", "reason_code", "action",
     "impressions_90d", "days_since_last_update", "avg_position", "ctr", "trend_direction"]
]

flagged = queue[queue["action"]=="review_for_refresh"]
print("Total flagged for review:", len(flagged))
print("Unique clients among flagged:", flagged["client_id"].nunique())
flagged.head(17)

Total flagged for review: 17
Unique clients among flagged: 4


,content_id,client_id,baseline_score,reason_code,action,impressions_90d,days_since_last_update,avg_position,ctr,trend_direction
16751,content_cf56e2e2e282,client_7f2253d7e2,61678,stale_visible_page,review_for_refresh,61678,194,19.7,0.15,down
16514,content_7368877ea310,client_7f2253d7e2,59472,stale_visible_page,review_for_refresh,59472,194,24.8,0.13,down
7021,content_1bfaa38ff26c,client_7f2253d7e2,25715,stale_visible_page,review_for_refresh,25715,194,22.2,0.23,down
21268,content_0a91db491d14,client_7f2253d7e2,13299,stale_visible_page,review_for_refresh,13299,193,10.5,0.49,down
11489,content_5feee3994adb,client_7f2253d7e2,7812,stale_visible_page,review_for_refresh,7812,194,39.0,0.01,down
12045,content_c2d929d83eaa,client_7f2253d7e2,7558,stale_visible_page,review_for_refresh,7558,193,17.9,0.20,down
698,content_b16bd7307b39,client_7f2253d7e2,4590,stale_visible_page,review_for_refresh,4590,194,31.0,0.00,down
5327,content_fe16a55cd13d,client_7f2253d7e2,4556,stale_visible_page,review_for_refresh,4556,194,16.4,0.33,down
26810,content_ecb6215e79fd,client_7f2253d7e2,4429,stale_visible_page,review_for_refresh,4429,194,25.3,0.38,down
20837,content_928af3e22c80,client_7f2253d7e2,1697,stale_visible_page,review_for_refresh,1697,193,15.8,0.12,down


17 pages are flagged review_for_refresh, reason code stale_visible_page (both 180+ days stale and 500+ impressions in the last 90 days). In plain words for a human reviewer: "This page still gets meaningful traffic but hasn't been touched in over 6 months — check whether it needs an update before it loses more ground." 12 of the 17 (71%) belong to a single client, a pattern worth flagging to a reviewer as a possible scoring-formula bias toward high-traffic clients rather than a genuine finding that one client's content is uniquely neglected.

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

Who uses this: a content strategist or SEO reviewer with limited weekly capacity, using this queue to decide which pages to look at first, not which pages to auto-edit.

Where it stops being valid: this queue is built on a single-window snapshot (90-day trailing window) and a same-window proxy label — it does not account for seasonality, recent site-wide changes, or client-specific context (e.g. a planned redesign). It should be re-generated regularly, not treated as a one-time static list. It also should not be used to compare urgency ACROSS clients, since the scoring formula uses raw impression counts, which structurally favors high-traffic clients (as seen in the 12/17 concentration above).

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

Before acting on any flagged page, a human should check: (1) is this page intentionally evergreen/rarely-updated by design? (2) is the "decline" actually seasonal or due to a sibling page absorbing traffic (consolidation), rather than a real quality problem? (3) does avg_position/ctr already look healthy despite staleness — if so, a refresh may not be the priority.

No-go list — this should NEVER be automated: (1) auto-publishing content changes without human review — this queue only prioritizes REVIEW, not action; (2) using this score to justify removing/de-indexing a page — that requires much stronger evidence than a same-window proxy label; (3) comparing raw scores across clients to judge relative team performance, since the formula is not normalized per client; (4) treating "declining" as proof a refresh will fix anything — no causal claim is made or supported by this data.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

Signals that these recommendations have gone stale: (1) the underlying data snapshot is more than ~90 days old, since the core signals are trailing-90-day windows; (2) a large swing in a client's overall traffic (e.g. a site migration or algorithm update) that would invalidate the baseline assumptions the score was tuned on; (3) if a future validation run (like the honest-split audit in ML-09) shows Precision@50 dropping meaningfully below the ~0.66 currently observed, indicating the model/rule has drifted from the data it was built on; (4) any change to the underlying dataset schema or available signals (e.g. new fields becoming available, as would happen moving from the starter dataset to the full warehouse).

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
queue.to_csv("work/outputs/action_playbook_queue.csv", index=False)
flagged.to_csv("work/outputs/action_playbook_flagged.csv", index=False)
print("Exported", len(queue), "total rows and", len(flagged), "flagged rows to work/outputs/")

Exported 30000 total rows and 17 flagged rows to work/outputs/


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.